In [1]:
# -----------------------------
# Indlæsning af data
# -----------------------------

import pandas as pd
import numpy as np

# Indlæs draft og Elo data
draft = pd.read_csv("Data/draft_encoded.csv")
elo = pd.read_csv("Data/elo_features.csv")

# Print størrelse på datasæt (rækker, kolonner)
print("Draft:", draft.shape)
print("Elo:", elo.shape)

Draft: (2255, 3490)
Elo: (2079, 8)


In [2]:
# -----------------------------
# Merge draft og Elo
# -----------------------------

elo = elo[["gameid", "elo_diff"]]

data = draft.merge(elo, on="gameid")

print("Merged:", data.shape)

Merged: (2041, 3491)


In [3]:
print(data.columns)

Index(['firstPick_blue', 'blue_side', 'gameid_13060-13060_game_1',
       'gameid_13060-13060_game_2', 'gameid_13060-13060_game_3',
       'gameid_13061-13061_game_1', 'gameid_13061-13061_game_2',
       'gameid_13062-13062_game_1', 'gameid_13062-13062_game_2',
       'gameid_13063-13063_game_1',
       ...
       'pick5_red_Zac', 'pick5_red_Zed', 'pick5_red_Zeri', 'pick5_red_Ziggs',
       'pick5_red_Zilean', 'pick5_red_Zoe', 'pick5_red_Zyra', 'result',
       'gameid', 'elo_diff'],
      dtype='str', length=3491)


In [4]:
# -----------------------------
# Feature engineering
# -----------------------------

# Interaction feature (kombinerer Elo og side)
data["elo_x_side"] = data["elo_diff"] * data["blue_side"]

C:\Users\mikke\AppData\Local\Temp\ipykernel_35792\122094901.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["elo_x_side"] = data["elo_diff"] * data["blue_side"]


In [5]:
# -----------------------------
# Split features og target
# -----------------------------

# Features (alle kolonner undtagen target og id)
X = data.drop(columns=["result", "gameid"])

# Target (det vi vil forudsige)
y = data["result"]

# Print størrelse på feature matrix
print("X shape:", X.shape)

X shape: (2041, 3490)


In [6]:
# -----------------------------
# Check for fejl
# -----------------------------

# Tjek for tekst (skal ikke være i modellen)
print("String columns:", X.select_dtypes(include="object").columns)

# Tjek for manglende værdier (NaN)
print("NaN values:", X.isna().sum().sum())

String columns: Index([], dtype='str')
NaN values: 0


In [7]:
# -----------------------------
# Train/Test split
# -----------------------------

from sklearn.model_selection import train_test_split

# Split data i train og test (80/20)
# shuffle=False → bevarer tidsrækkefølge
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# Print størrelser
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (1632, 3490)
Test: (409, 3490)


In [8]:
# -----------------------------
# Scaling
# -----------------------------

from sklearn.preprocessing import StandardScaler

# Opret scaler
scaler = StandardScaler()

# Fit på train og skalér data
X_train_scaled = scaler.fit_transform(X_train)

# Brug samme scaler på test data
X_test_scaled = scaler.transform(X_test)

In [9]:
# -----------------------------
# Logistic Regression
# -----------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Opret model
model = LogisticRegression(max_iter=1000)

# Træn model på train data
model.fit(X_train_scaled, y_train)

# Forudsig sandsynligheder på test data
preds = model.predict_proba(X_test_scaled)[:, 1]

# Evaluer model (AUC score)
print("AUC:", roc_auc_score(y_test, preds))

AUC: 0.526676279740447
